# SGL narrow test (official repo, tiny split)

This notebook **clones [NUS-HPC-AI-Lab/SGL](https://github.com/NUS-HPC-AI-Lab/SGL)** and runs their **`eval/vqa/evaluate_vqa.py`** on a **hand-built mini dataset** wired through the existing **`ocrvqa_val`** path (exact-match accuracy, no full TextVQA annotation file).

**Why `torchrun`?** Upstream `evaluate_vqa.py` calls `torch.distributed.barrier` / `all_gather` even for a single GPU. `torchrun --nproc_per_node=1` sets `RANK` / `WORLD_SIZE` / `LOCAL_RANK` so that path works.

**Models:** defaults use **InternVL2-2B → InternVL2-8B** (fits comfortably in **80GB** with both loaded in bf16). You can switch `LARGE_MODEL` to `OpenGVLab/InternVL2-26B` if you accept higher VRAM use.

In [ ]:
# --- User config ---
import os

SGL_REPO = "https://github.com/NUS-HPC-AI-Lab/SGL.git"
WORKDIR = os.environ.get("SGL_WORKDIR", "/content")
SGL_ROOT = os.path.join(WORKDIR, "SGL")

# Hugging Face model ids (will be downloaded to HF cache; symlinks passed as local dirs)
SMALL_MODEL = "OpenGVLab/InternVL2-2B"
LARGE_MODEL = "OpenGVLab/InternVL2-8B"  # try "OpenGVLab/InternVL2-26B" on 80GB if desired

NUM_SAMPLES = 8  # tiny VQA split
PRUNE_LAYER = 0.4   # upstream: --large_model_prune_layer
PRUNE_RATIO = 0.4   # upstream: --large_model_prune_ratio

os.makedirs(WORKDIR, exist_ok=True)
print("WORKDIR:", WORKDIR)
print("SGL_ROOT:", SGL_ROOT)

## 1) Optional: Hugging Face token
Only needed if a checkpoint is gated. In Colab: **Secrets** → `HF_TOKEN`, or run `huggingface-cli login` once.

In [ ]:
import os
from huggingface_hub import login

tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if tok:
    login(token=tok, add_to_git_credential=False)
    print("HF_TOKEN found: logged in.")
else:
    print("No HF_TOKEN in env; public models should still download.")

## 2) Clone SGL + Python deps

Pinned versions follow **InternVL chat** requirements where possible. Colab may already ship **PyTorch 2.10**; we avoid reinstalling `torch` / `torchvision` unless imports fail.

In [ ]:
import os
import subprocess
import sys

def sh(cmd: str) -> None:
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")

if not os.path.isdir(SGL_ROOT):
    sh(f"git clone --depth 1 {SGL_REPO} {SGL_ROOT}")
else:
    print("SGL_ROOT exists; skip clone.")

# Core deps for InternVL2 + SGL eval (flash-attn not required: ViT falls back to naive attention)
sh(
    sys.executable
    + " -m pip install -q --upgrade pip && "
    + sys.executable
    + " -m pip install -q "
    "transformers==4.37.2 timm==0.9.12 einops==0.6.1 einops-exts==0.0.4 "
    "peft==0.10.0 sentencepiece==0.1.99 accelerate huggingface_hub "
    "tqdm pyyaml opencv-python scipy protobuf packaging"
)

import importlib
for m in ("transformers", "timm", "einops", "peft"):
    importlib.import_module(m)
    print("OK", m)

## 3) Tiny dataset → `data/ocrvqa/` (reuses `ocrvqa_val` in `evaluate_vqa.py`)

Each line is JSONL: `image`, `question`, `question_id`, `answer` (string label for exact match).

In [ ]:
import json
import os
from PIL import Image, ImageDraw

data_root = os.path.join(SGL_ROOT, "data", "ocrvqa")
img_dir = os.path.join(data_root, "toy_images")
os.makedirs(img_dir, exist_ok=True)

def make_image(path: str, color: tuple[int, int, int]) -> None:
    im = Image.new("RGB", (512, 384), color)
    d = ImageDraw.Draw(im)
    d.rectangle([160, 120, 352, 264], outline=(255, 255, 255), width=4)
    im.save(path)

samples = [
    ("toy_000.jpg", (180, 40, 40), "What is the dominant color in this image?", "red"),
    ("toy_001.jpg", (40, 140, 40), "What is the dominant color in this image?", "green"),
    ("toy_002.jpg", (40, 80, 200), "What is the dominant color in this image?", "blue"),
    ("toy_003.jpg", (30, 30, 30), "Is the image mostly dark? Answer yes or no.", "yes"),
    ("toy_004.jpg", (240, 240, 240), "Is the image mostly bright? Answer yes or no.", "yes"),
    ("toy_005.jpg", (200, 180, 40), "Name one color visible in the image.", "yellow"),
    ("toy_006.jpg", (120, 40, 140), "Name one color visible in the image.", "purple"),
    ("toy_007.jpg", (40, 160, 160), "Name one color visible in the image.", "cyan"),
][:NUM_SAMPLES]

rows = []
for i, (name, color, q, a) in enumerate(samples):
    ip = os.path.join(img_dir, name)
    make_image(ip, color)
    rows.append(
        {
            "image": os.path.abspath(ip),
            "question": q,
            "question_id": int(i),
            "answer": a,
        }
    )

train_path = os.path.join(data_root, "ocrvqa_train.jsonl")
val_path = os.path.join(data_root, "ocrvqa_val.jsonl")
for p in (train_path, val_path):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

print("Wrote:", val_path, "lines:", len(rows))

## 4) Materialize HF checkpoints as local folders (what the shell script expects)

`evaluate_vqa.py` loads configs with `InternVLChatConfig.from_json_file(f"{ckpt}/config.json")`, so we pass **local snapshot directories**.

In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

ckpt_root = os.path.join(WORKDIR, "model_ckpts")
os.makedirs(ckpt_root, exist_ok=True)

small_snap = snapshot_download(SMALL_MODEL, cache_dir=os.path.join(ckpt_root, "hf_cache"))
large_snap = snapshot_download(LARGE_MODEL, cache_dir=os.path.join(ckpt_root, "hf_cache"))

# Stable directory names (upstream logging uses checkpoint path tokens; symlinks keep this readable)
small_dir = os.path.join(ckpt_root, "small_ckpt")
large_dir = os.path.join(ckpt_root, "large_ckpt")
for dst, src in ((small_dir, small_snap), (large_dir, large_snap)):
    if os.path.islink(dst) or os.path.isfile(dst):
        os.remove(dst)
    elif os.path.isdir(dst) and not os.path.islink(dst):
        shutil.rmtree(dst)
    os.symlink(os.path.abspath(src), dst, target_is_directory=True)

print("small_checkpoint:", small_dir, "->", os.path.realpath(small_dir))
print("large_checkpoint:", large_dir, "->", os.path.realpath(large_dir))

## 5) Run upstream eval (single GPU)

This mirrors `textvqa2B-26B.sh` but uses **`ocrvqa_val`** and your local checkpoints.

**If something breaks:**
- After `pip install`, **restart the runtime** once if `import transformers` still sees an old version.
- `Default process group has not been initialized`: re-run the **torchrun** cell (do not call `evaluate_vqa.py` with plain `python`).
- `AssertionError` on `image_size` / `use_thumbnail`: pick another model pair (2B/8B/26B should match within the same InternVL2 generation).
- CUDA OOM on **26B**: use **8B** as `LARGE_MODEL`, or add `--load-in-8bit` / `--auto` to the command in the next cell (requires compatible `bitsandbytes` on your image).

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONPATH"] = SGL_ROOT + os.pathsep + env.get("PYTHONPATH", "")
env["CUDA_VISIBLE_DEVICES"] = "0"

cmd = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nproc_per_node",
    "1",
    os.path.join(SGL_ROOT, "eval", "vqa", "evaluate_vqa.py"),
    "--small_checkpoint",
    small_dir,
    "--large_checkpoint",
    large_dir,
    "--datasets",
    "ocrvqa_val",
    "--dynamic",
    "--out-dir",
    os.path.join(SGL_ROOT, "results_colab_narrow"),
    "--large_model_prune_layer",
    str(PRUNE_LAYER),
    "--large_model_prune_ratio",
    str(PRUNE_RATIO),
    "--num-workers",
    "0",
]

print("Running:\n", " ".join(cmd), flush=True)
r = subprocess.run(cmd, cwd=SGL_ROOT, env=env)
print("exit code:", r.returncode)
if r.returncode != 0:
    raise RuntimeError("evaluate_vqa.py failed; scroll up for the traceback.")

## 6) (Optional) second run, more aggressive pruning
Uncomment to compare a different `--large_model_prune_ratio` without re-running the whole notebook.

In [ ]:
# PRUNE_LAYER = 0.4
# PRUNE_RATIO = 0.2
# # re-run the previous cell after editing PRUNE_*